In [1]:
#%% import libraries

import os
import sys
import time
from time import sleep
import  threading
from multiprocessing import Queue

from IPython.display import clear_output

from pypylon import pylon
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

import datetime


In [38]:

# %% def the function make_acquisition (for a given frequency, for all chosen cameras)
def make_acquisition(freq_acq, cam_array, list_camera_SN, acquisition_folder, save_images=True,numberOfImagesToGrab=100):
    
    try:
        cam_array.Open()

        # demonstrate some feature access
        list_str_camera_serial_number = []
        #list_freq_acq_effective = []
        for idx_cam in range(len(list(cam_array))):
            camera = cam_array[idx_cam]
            print(camera.Width.GetInc())
            print(camera.Width.GetValue())

            camera.AcquisitionFrameRate.SetValue(freq_acq)  # Replace with your desired frame rate
            camera.AcquisitionFrameRateEnable.SetValue(True)            
            print('frame rate : ',camera.ResultingFrameRate.GetValue())

            # mettre un buffer assez grand :
            camera.MaxNumBuffer.SetValue(9000)
            str_camera_serial_number = camera.DeviceSerialNumber()
            list_str_camera_serial_number.append(str_camera_serial_number)
            #list_freq_acq_effective.append(freq_acq_effective)
            savefolder = f'{acquisition_folder}/camera_{str_camera_serial_number}/{str(np.round(freq_acq,4))}Hz/'

            ## create directory to save images
            if save_images:
                if os.path.exists(acquisition_folder)==False:
                    os.mkdir(acquisition_folder)
                if os.path.exists(acquisition_folder+'camera_'+str_camera_serial_number+'/')==False:
                    os.mkdir(acquisition_folder+'camera_'+str_camera_serial_number+'/')
                if os.path.exists(savefolder)==False:
                    os.mkdir(savefolder)
                if os.path.exists(savefolder + 'images/')==False:
                    os.mkdir(savefolder + 'images/')
            ## write camera settings in a txt file    
            file = open(savefolder + 'camera_settings.txt','w')
            file.write('Serial Number : '+ list_camera_SN[idx_cam]+' \n')
            file.write('Exposure Time : ' + str(camera.ExposureTime.GetValue())+' microseconds \n')
            file.write('Width : ' + str(camera.Width.GetValue())+' \n')  # Replace with your desired width
            file.write('Height : '+str(camera.Height.GetValue())+' \n')  # Replace with your desired height
            file.write('OffsetX : '+str(camera.OffsetX.GetValue())+' \n')  # Replace with your desired X offset
            file.write('OffsetY : '+str(camera.OffsetY.GetValue())+' \n')  # Replace with your desired Y offset
            file.write('Max Buffer Size : '+str(camera.MaxNumBuffer.GetValue())+' \n')
            file.close()
            
        # note the date and time of the beginning of the acquisition (useful in file names)
        time_acquisition = datetime.datetime.now()
        year = time_acquisition.year
        month = time_acquisition.month
        day = time_acquisition.day
        hour = time_acquisition.hour
        minutes = time_acquisition.minute
        seconds = time_acquisition.second
        str_time_acquisition = str(year)+str(month).zfill(2)+str(day).zfill(2)+'_'+str(hour).zfill(2)+str(minutes).zfill(2)+str(seconds).zfill(2)
        
        
        # start to grab images for all cameras at the same time
        #cam_array.StartGrabbing(numberOfImagesToGrab)
        array_time_elapsed = []
        time_init = time.time()
        for camera in cam_array:
            camera.StartGrabbing()
            t1=time.time()
            array_time_elapsed.append(t1 - time_init)
        
        file_path = f'{savefolder}/t_init.txt'

        # Écrire dans le fichier
        with open(file_path, "a") as f:  # Mode "append" pour ajouter à la fin
            f.write(f"{time_init}\t{t1}\n")  # Écrire les 2 éléments séparés par une tabulation

        
        array_time_elapsed = np.array(array_time_elapsed)
        print(array_time_elapsed)
        array_time_elapsed_wrtcam0 = array_time_elapsed - array_time_elapsed[0]
        array_num_images_elapsed = array_time_elapsed_wrtcam0//(1/freq_acq)
        print(array_time_elapsed_wrtcam0)
        print(array_num_images_elapsed)
            
        i = 0

        while cam_array[0].IsGrabbing(): # on a dû adapter :/ (normalement c'était juste cam_array.IsGrabbing)
            for idx_cam in range(len(list(cam_array))):
                camera = cam_array[idx_cam]
                grabResult = camera.RetrieveResult(5000, pylon.TimeoutHandling_ThrowException)
                if grabResult.GrabSucceeded():
                    # Access the image data.
                    print('idx_cam =',idx_cam)
                    #print("SizeX: ", grabResult.Width)
                    #print("SizeY: ", grabResult.Height)
                    img = grabResult.Array
                    #print("Gray value of first pixel: ", img[0, 0])
                    if save_images==True:
                        # Convert the NumPy array to a PIL Image object
                        image = Image.fromarray(img)
                        # Save the image as a TIFF file (black and white)
                        str_camera_serial_number = list_str_camera_serial_number[idx_cam]
                        savefolder = acquisition_folder+'camera_'+str_camera_serial_number+'/' + str(np.round(freq_acq,4))+'Hz/'

                        namefile = 'Basler__'+str_camera_serial_number+'__'+str_time_acquisition+'_'+str(int(i+array_num_images_elapsed[idx_cam]-array_num_images_elapsed[-1])).zfill(4)+'.tiff' # pour les cameras qui commencent avant les autres, les premieres images auront des nombres négatifs
                        #namefile = 'Basler__'+str_camera_serial_number+'__'+str_time_acquisition+'_'+str(int(i)).zfill(4)+'.tiff'
                        if (int(i+array_num_images_elapsed[idx_cam]-array_num_images_elapsed[-1]) < 0)|(int(i+array_num_images_elapsed[idx_cam]-array_num_images_elapsed[-1])>=numberOfImagesToGrab):
                            pass
                        else:
                            image.save(savefolder+'images/'+namefile)
                            print('image number '+str(i))
                            clear_output(wait=True)
                            
                        
            grabResult.Release()
            print(i)
            i+=1
            if i>=numberOfImagesToGrab + array_num_images_elapsed[-1]:
                cam_array.StopGrabbing()
                cam_array.Close()            

    finally:
        # Ensure camera is closed properly
        #cam_array.StopGrabbing()
        cam_array.Close()



In [39]:
######################################################
# SCAN DES DISPOSITIFS USB POUR TROUVER LA/LES CAMERAS
######################################################

# Create a CameraFactory instance
tl_factory = pylon.TlFactory.GetInstance()

# Choose the devices to use
device_count = tl_factory.EnumerateDevices()
NUM_MAX_CAMERAS = len(list(device_count))

list_camera_SN = ['40232065']

#cam_array_temp = pylon.InstantCameraArray(NUM_MAX_CAMERAS)
NUM_CAMERAS = len(list_camera_SN)
cam_array = pylon.InstantCameraArray(NUM_CAMERAS)

cam_info = tl_factory.GetInstance().EnumerateDevices()
selected_cameras_indices = []
for i in range(len(list(cam_info))):
    camera = cam_info[i]
    if camera.GetSerialNumber() in list_camera_SN:
        selected_cameras_indices.append(i)

#print(selected_cameras_indices)

j=0
for i in selected_cameras_indices:
    cam_array[j].Attach(tl_factory.CreateDevice(device_count[i]))
    cam_array[j].Open()
    print('camera with index '+str(j) + ' has seral number : '+ cam_array[j].DeviceSerialNumber())
    cam_array[j].Close()
    j+=1

if len(device_count)==0:
    print('0 camera found')


#############################################################################################################################




camera with index 0 has seral number : 40232065


In [40]:


ddd = datetime.datetime.now()
date = str(ddd.year) + str(ddd.month).zfill(2) + str(ddd.day).zfill(2)

datefolder = f'C:/Users/Vasco Zanchi/Desktop/CASCADE26/tests_interfacage_test_bresilien/{date}/'
dailyfolder = f'{datefolder}/tests_bresilens/'

if os.path.exists(datefolder)==False:
    os.mkdir(datefolder)
if os.path.exists(dailyfolder)==False:
    os.mkdir(dailyfolder)



num_acquisition = 1
acquisition_folder = dailyfolder + 'Acquisition_' + str(num_acquisition) + '/'
while ((os.path.exists(acquisition_folder))|(os.path.exists(f'{dailyfolder}/Acquisition_{str(num_acquisition)}/'))):
    num_acquisition += 1
    acquisition_folder = dailyfolder + 'Acquisition_' + str(num_acquisition) + '/'

print("Acquisition number : "+str(num_acquisition))


##################################################### INPUTS & SETTINGS ######################################################
freq_acq = 15

# %% choose number of images to grab and the name of the daily folder
numberOfImagesToGrab = 9000 # choisir nombre d'images
save_images = True

Acquisition number : 5


## Test d'acquisition d'images

In [41]:
make_acquisition(20, cam_array=cam_array, list_camera_SN=list_camera_SN, acquisition_folder=acquisition_folder, save_images=True, numberOfImagesToGrab=10)

9


## Test lecture data PhidgetBridge

In [7]:
dailyfolder

'C:/Users/Vasco Zanchi/Desktop/CASCADE26/tests_interfacage_test_bresilien/20261020//tests_bresilens/'

In [10]:
from icewave.vasco.interfacage.PhigetBridge import interface_phidget_bridge_4inputs
%matplotlib qt

params_phidget = interface_phidget_bridge_4inputs.create_default_dict_param()
params_phidget['channels2plot'] = [0]
params_phidget['save'] = True
params_phidget['name_output_file'] = f'{dailyfolder}/PhidgetBridge/test_interface_brazilian_test.txt'

interface_phidget_bridge_4inputs.main(params=params_phidget)


output file already exists, adding in that file..
number of points on the figure :  1
number of points on the figure :  2
number of points on the figure :  3
number of points on the figure :  4
number of points on the figure :  5
number of points on the figure :  6
number of points on the figure :  7
number of points on the figure :  8
number of points on the figure :  9
number of points on the figure :  10
number of points on the figure :  11
number of points on the figure :  12
number of points on the figure :  13
number of points on the figure :  14
number of points on the figure :  15
number of points on the figure :  16
number of points on the figure :  17
number of points on the figure :  18
number of points on the figure :  19
number of points on the figure :  20
number of points on the figure :  21
number of points on the figure :  22
number of points on the figure :  23
number of points on the figure :  24
number of points on the figure :  25
number of points on the figure :  

KeyboardInterrupt: 

In [16]:
freq_acq

15

### Lancer caméra et capteur de force en même temps (avec même fréquence d'acquisition)

In [ ]:
try:

    t0 = time.time()

    acquisition_thread = threading.Thread(
        target=make_acquisition,
        args=(freq_acq, cam_array, list_camera_SN, dailyfolder, True, 1000),
        daemon=True
    )
    t1 = time.time()

    acquisition_thread.start()
    
    t2 = time.time()
    interface_phidget_bridge_4inputs.main(params=params_phidget)

except KeyboardInterrupt:
    print("\nArrêt du programme...")

finally:
    print("Arrêt de l'acquisition...")






999


In [23]:
dailyfolder

'C:/Users/Vasco Zanchi/Desktop/CASCADE26/tests_interfacage_test_bresilien/20261020//tests_bresilens/'